In [1]:
import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Start spark
spark = SparkSession.builder.appName("EDA_Detective").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Load CSVs
df_attr = spark.read.csv("data/features_attributes.csv", header=True, inferSchema=True)
df_fin = spark.read.csv("data/features_financials.csv", header=True, inferSchema=True)
df_click = spark.read.csv("data/feature_clickstream.csv", header=True, inferSchema=True)

print("Data loaded successfully.")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/15 16:43:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Data loaded successfully.


In [2]:
from pyspark.sql.functions import col, length, trim

print("--- PHASE 1: CUSTOMER ID FORENSICS ---")

# 1. Check for Duplicates (If Total Rows > Unique IDs, we have duplicates)
print("Clickstream - Total Rows:", df_click.count(), "| Unique IDs:", df_click.select("Customer_ID").distinct().count())
print("Financials  - Total Rows:", df_fin.count(), "| Unique IDs:", df_fin.select("Customer_ID").distinct().count())
print("Attributes  - Total Rows:", df_attr.count(), "| Unique IDs:", df_attr.select("Customer_ID").distinct().count())

# 2. Check for Missing/Null IDs
print("\nMissing/Null IDs in Attributes:", df_attr.filter(col("Customer_ID").isNull() | (col("Customer_ID") == "")).count())

# 3. Check for Hidden Spaces (Trailing/Leading whitespace)
df_attr_spaces = df_attr.withColumn("raw_len", length(col("Customer_ID"))) \
                        .withColumn("trim_len", length(trim(col("Customer_ID")))) \
                        .filter(col("raw_len") != col("trim_len"))
print("IDs with hidden whitespace in Attributes:", df_attr_spaces.count())

--- PHASE 1: CUSTOMER ID FORENSICS ---


Clickstream - Total Rows: 215376 | Unique IDs: 8974
Financials  - Total Rows: 12500 | Unique IDs: 12500
Attributes  - Total Rows: 12500 | Unique IDs: 12500

Missing/Null IDs in Attributes: 0
IDs with hidden whitespace in Attributes: 0


In [3]:
print("--- PHASE 2: SCHEMA & UNDERSCORE TRAPS ---")

# 1. Check the Schemas
print("Attributes Schema:")
df_attr.printSchema() 
# Look closely at 'Age'. Is it a string?

print("Financials Schema:")
df_fin.printSchema()
# Look closely at 'Annual_Income' and 'Num_of_Loan'. Are they strings?

# 2. Hunt for the Underscores specifically
print("\nHunting for underscores in Attributes (Age):")
df_attr.filter(col("Age").like("%_%")).show(5)

print("\nHunting for underscores in Financials (Annual Income):")
df_fin.filter(col("Annual_Income").like("%_%")).select("Customer_ID", "Annual_Income").show(5)

--- PHASE 2: SCHEMA & UNDERSCORE TRAPS ---
Attributes Schema:
root
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- SSN: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- snapshot_date: date (nullable = true)

Financials Schema:
root
 |-- Customer_ID: string (nullable = true)
 |-- Annual_Income: string (nullable = true)
 |-- Monthly_Inhand_Salary: double (nullable = true)
 |-- Num_Bank_Accounts: integer (nullable = true)
 |-- Num_Credit_Card: integer (nullable = true)
 |-- Interest_Rate: integer (nullable = true)
 |-- Num_of_Loan: string (nullable = true)
 |-- Type_of_Loan: string (nullable = true)
 |-- Delay_from_due_date: integer (nullable = true)
 |-- Num_of_Delayed_Payment: string (nullable = true)
 |-- Changed_Credit_Limit: string (nullable = true)
 |-- Num_Credit_Inquiries: double (nullable = true)
 |-- Credit_Mix: string (nullable = true)
 |-- Outstanding_Debt: string (nullable = true)
 |

In [4]:
print("--- PHASE 3: OUTLIER DETECTION ---")

# Note: Because the professor poisoned Age and Income with strings, 
# describe() won't work perfectly on them yet. But we can check the others!

# Check numerical boundaries for Financials
df_fin.select(
    "Monthly_Inhand_Salary", 
    "Num_Bank_Accounts", 
    "Interest_Rate", 
    "Num_Credit_Card"
).describe().show()

--- PHASE 3: OUTLIER DETECTION ---
+-------+---------------------+------------------+------------------+------------------+
|summary|Monthly_Inhand_Salary| Num_Bank_Accounts|     Interest_Rate|   Num_Credit_Card|
+-------+---------------------+------------------+------------------+------------------+
|  count|                12500|             12500|             12500|             12500|
|   mean|   4188.5923027131585|          16.93992|          73.21336|          23.17272|
| stddev|   3180.1476109204173|114.35081529709485|468.68222710052765|132.00586642912222|
|    min|    303.6454166666666|                -1|                 1|                 0|
|    max|   15204.633333333331|              1756|              5789|              1499|
+-------+---------------------+------------------+------------------+------------------+



In [5]:
import pyspark.sql.functions as F

print("--- PHASE 4: MISSING DATA COUNT ---")

# Count nulls for every column in Financials
df_fin.select([
    F.sum(F.when(col(c).isNull(), 1).otherwise(0)).alias(c) 
    for c in df_fin.columns
]).show(vertical=True)

--- PHASE 4: MISSING DATA COUNT ---
-RECORD 0------------------------
 Customer_ID              | 0    
 Annual_Income            | 0    
 Monthly_Inhand_Salary    | 0    
 Num_Bank_Accounts        | 0    
 Num_Credit_Card          | 0    
 Interest_Rate            | 0    
 Num_of_Loan              | 0    
 Type_of_Loan             | 1426 
 Delay_from_due_date      | 0    
 Num_of_Delayed_Payment   | 0    
 Changed_Credit_Limit     | 0    
 Num_Credit_Inquiries     | 0    
 Credit_Mix               | 0    
 Outstanding_Debt         | 0    
 Credit_Utilization_Ratio | 0    
 Credit_History_Age       | 0    
 Payment_of_Min_Amount    | 0    
 Total_EMI_per_month      | 0    
 Amount_invested_monthly  | 0    
 Payment_Behaviour        | 0    
 Monthly_Balance          | 0    
 snapshot_date            | 0    



In [6]:
print("--- PHASE 5: CATEGORICAL CONSISTENCY ---")

# Check all the unique labels in Credit_Mix
print("Credit Mix Categories:")
df_fin.groupBy("Credit_Mix").count().orderBy("count").show(truncate=False)

# Check Occupations
print("\nOccupation Categories:")
df_attr.groupBy("Occupation").count().orderBy("count").show()

--- PHASE 5: CATEGORICAL CONSISTENCY ---
Credit Mix Categories:
+----------+-----+
|Credit_Mix|count|
+----------+-----+
|Bad       |2360 |
|_         |2611 |
|Good      |3032 |
|Standard  |4497 |
+----------+-----+


Occupation Categories:
+-------------+-----+
|   Occupation|count|
+-------------+-----+
|       Writer|  728|
|      Manager|  736|
|     Musician|  741|
|       Doctor|  760|
|   Journalist|  761|
| Entrepreneur|  776|
|Media_Manager|  780|
|    Developer|  780|
|     Mechanic|  780|
|      Teacher|  782|
|    Scientist|  789|
|   Accountant|  791|
|     Engineer|  793|
|    Architect|  795|
|       Lawyer|  828|
|      _______|  880|
+-------------+-----+



In [7]:
import pyspark.sql.functions as F

print("--- DEEP SCAN: FAKE NULLS & BLANKS ---")

# A list of common "poison" strings used to represent missing data
suspicious_strings = ["NM", "N/A", "NA", "null", "NULL", "-", "None", ""]

# 1. Scan Attributes Table for Fake Nulls or pure whitespace
print("Scanning Attributes...")
for column_name in df_attr.columns:
    count = df_attr.filter(
        F.col(column_name).isin(suspicious_strings) | 
        F.col(column_name).rlike(r"^\s+$")  # Matches strings that are purely blank spaces
    ).count()
    if count > 0:
        print(f"  WARNING: Column '{column_name}' contains {count} hidden blanks/fake nulls.")

# 2. Scan Financials Table for Fake Nulls
print("\nScanning Financials...")
for column_name in df_fin.columns:
    count = df_fin.filter(
        F.col(column_name).isin(suspicious_strings) | 
        F.col(column_name).rlike(r"^\s+$")
    ).count()
    if count > 0:
        print(f"  WARNING: Column '{column_name}' contains {count} hidden blanks/fake nulls.")

--- DEEP SCAN: FAKE NULLS & BLANKS ---
Scanning Attributes...

Scanning Financials...


In [8]:
print("--- DEEP SCAN: CONCATENATION & LENGTH OUTLIERS ---")

# Find the longest strings in the Attributes table
print("Longest Strings in Attributes:")
df_attr.select(
    F.max(F.length("Occupation")).alias("Max_Occupation_Length"),
    F.max(F.length("Age")).alias("Max_Age_Length"),
    F.max(F.length("Name")).alias("Max_Name_Length")
).show()

# If the max length looks suspiciously high (e.g., Age length > 3), let's look at the actual rows
print("Looking at the weirdest/longest Age values:")
df_attr.withColumn("len", F.length("Age")) \
       .orderBy(F.desc("len")) \
       .select("Customer_ID", "Age", "len") \
       .show(5)

--- DEEP SCAN: CONCATENATION & LENGTH OUTLIERS ---
Longest Strings in Attributes:
+---------------------+--------------+---------------+
|Max_Occupation_Length|Max_Age_Length|Max_Name_Length|
+---------------------+--------------+---------------+
|                   13|             5|             25|
+---------------------+--------------+---------------+

Looking at the weirdest/longest Age values:
+-----------+-----+---+
|Customer_ID|  Age|len|
+-----------+-----+---+
| CUS_0x1b1a|8669_|  5|
| CUS_0x4423|7723_|  5|
| CUS_0x24e1|2329_|  5|
| CUS_0x13e4|1248_|  5|
| CUS_0x2637|5798_|  5|
+-----------+-----+---+
only showing top 5 rows



In [9]:
print("--- DEEP SCAN: WEIRD CHARACTERS ---")

# Scan Occupation for anything that isn't a letter, number, space, or standard punctuation
print("Suspicious Occupations:")
df_attr.filter(F.col("Occupation").rlike(r"[^a-zA-Z0-9 \-_]")).select("Customer_ID", "Occupation").show(5)

# Scan Credit_Mix in Financials for weird symbols
print("Suspicious Credit Mixes:")
df_fin.filter(F.col("Credit_Mix").rlike(r"[^a-zA-Z0-9 \-_]")).select("Customer_ID", "Credit_Mix").show(5)

--- DEEP SCAN: WEIRD CHARACTERS ---
Suspicious Occupations:
+-----------+----------+
|Customer_ID|Occupation|
+-----------+----------+
+-----------+----------+

Suspicious Credit Mixes:
+-----------+----------+
|Customer_ID|Credit_Mix|
+-----------+----------+
+-----------+----------+

